# 线性回归：第一个可恢复训练任务

## 学习目标

能够从合成数据恢复线性参数，解释 MSE 梯度，并保存与恢复训练。


## 概念模型与执行路径

线性回归把训练过程缩小到一个权重和一个偏置，适合观察初始化、学习率、损失曲线和参数更新。


### 实验 1：生成带噪声数据并初始化回归模型

**实验目的**：构造一个真实参数已知的回归问题，并准备模型、优化器和损失函数。因为知道数据来自 $y=3x+2+\varepsilon$，训练后可以直接判断模型是否恢复了斜率 3 和截距 2。

`x` 由区间 `[-1, 1]` 上均匀分布的 128 个点组成。`unsqueeze(1)` 把形状从 `(128,)` 变为 `(128, 1)`，明确表示 128 个样本、每个样本 1 个特征。`y` 具有相同形状，并加入标准差约为 0.05 的高斯噪声，所以有限样本下的最优参数通常非常接近但不必精确等于 3 和 2。

`nn.Linear(1, 1)` 实现 $\hat y=xw+b$，内部只有形状 `(1, 1)` 的 weight 和形状 `(1,)` 的 bias。`torch.manual_seed(42)` 固定噪声和参数初始化，使结果可复现。SGD 每次沿负梯度方向更新参数；本页使用全批量训练，所以每轮梯度由全部 128 个样本计算。

`MSELoss` 默认返回所有样本平方误差的均值：$L=\frac{1}{N}\sum_i(\hat y_i-y_i)^2$。学习率 0.2 决定每次更新步幅。

**观察重点**：本单元只创建训练状态，尚未发生参数更新；此时模型预测主要由随机初始化决定。

In [ ]:
import torch
from torch import nn
torch.manual_seed(42)
x = torch.linspace(-1, 1, 128).unsqueeze(1)
y = 3 * x + 2 + torch.randn_like(x) * 0.05
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.2)
loss_fn = nn.MSELoss()


### 实验 2：执行全批量梯度下降并恢复参数

**实验目的**：运行 40 次完整参数更新，观察 MSE 下降以及 weight、bias 向真实参数收敛。每个 epoch 都使用同样的 128 个样本，因此不存在 mini-batch 采样噪声。

训练顺序为：

1. `zero_grad(set_to_none=True)` 清除上一轮累积梯度；
2. `model(x)` 计算 $\hat y$，`loss_fn` 计算标量 MSE；
3. `loss.backward()` 根据链式法则求 weight 和 bias 的梯度；
4. `optimizer.step()` 在 `no_grad` 语义下原地更新参数；
5. `loss.item()` 保存与计算图分离的 Python 数值。

对单特征模型，梯度为 $\partial L/\partial w=\frac{2}{N}\sum_i x_i(\hat y_i-y_i)$，$\partial L/\partial b=\frac{2}{N}\sum_i(\hat y_i-y_i)$。由于 x 关于 0 对称，斜率和截距的优化耦合较弱；最终参数应接近 `3.0` 和 `2.0`，loss 会下降到接近噪声方差 $0.05^2=0.0025$ 的水平。

**结果解读**：打印的是更新前第 1 轮和第 40 轮前向计算得到的 loss。最后一次 `step()` 后没有重新计算 loss，所以 `losses[-1]` 对应最后更新前的参数；差异在已收敛时通常很小。

In [ ]:
losses = []
for epoch in range(40):
    optimizer.zero_grad(set_to_none=True)
    loss = loss_fn(model(x), y)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
print("loss:", losses[0], "->", losses[-1])
print("learned:", model.weight.item(), model.bias.item())


### 实验 3：用损失曲线诊断收敛过程

**实验目的**：把每轮 MSE 可视化，判断优化是稳定收敛、速度过慢，还是发生振荡或发散。横轴是 epoch 索引，纵轴是该轮参数更新前计算出的 MSE。

在当前凸二次问题中，合适的固定学习率应产生快速下降并逐渐进入平台区。平台约为数据噪声造成的不可约训练误差；继续增加轮数不会把含噪数据的总体最优 MSE 降到严格的 0。若曲线上下剧烈振荡或不断增大，优先怀疑学习率过高；若几乎不动，则检查学习率、梯度和输入尺度。

**绘图提示**：当损失跨越多个数量级时，可改用 `plt.yscale('log')` 更清楚地观察后期收敛。Notebook 中 `plt.show()` 显式显示图像；在脚本或无图形环境中通常应改为 `plt.savefig(...)`。

In [ ]:
import matplotlib.pyplot as plt
plt.plot(losses)
plt.xlabel("epoch")
plt.ylabel("MSE")
plt.title("Linear regression convergence")
plt.show()


### 实验 4：使用终端脚本保存和恢复训练

**实验目的**：把 notebook 中的最小训练过程连接到可重复运行的命令行任务。这个单元只有命令示例，需要在仓库根目录的终端执行，不会在 notebook 内直接启动子进程。

第一次命令训练到 epoch 20，并把 checkpoint 写入默认的 `07-deep-learning/pytorch/artifacts/linear_regression.pt`。文件包含模型参数、SGD 状态和已完成的 epoch。第二次命令通过 `--resume` 加载该文件，把起始轮次设为 checkpoint epoch 加 1，再训练到 `--epochs 40`；因此这里的 `--epochs` 表示最终目标 epoch，而不是“额外训练 40 轮”。

脚本还会固定随机种子、自动选择 device，并使用 1024 个合成点；传 `--quick` 时改为 128 个点。恢复时脚本会重新生成相同数据，再加载模型和 optimizer。当前优化器是无 momentum 的 SGD，optimizer state 很少，但统一保存接口让代码可以安全扩展到 Adam 或带动量 SGD。

**注意**：如果 checkpoint 已经记录 epoch 40，再使用 `--epochs 40 --resume ...`，训练循环为空，但脚本仍会重新保存和打印参数。不要把不可信来源的 checkpoint 当成普通数据文件；本项目加载器使用 `weights_only=True` 降低反序列化风险。

In [ ]:
# 终端版本支持检查点和 --resume：
# python 07-deep-learning/pytorch/examples/linear_regression.py --epochs 20
# python 07-deep-learning/pytorch/examples/linear_regression.py --epochs 40 --resume 07-deep-learning/pytorch/artifacts/linear_regression.pt


## 底层机制

线性模型配合 MSE 得到关于参数的凸二次目标；只要数据矩阵提供足够信息，就不存在神经网络中常见的局部最优问题。Autograd 仍按通用计算图机制完成求导：Linear 保存反向所需输入，MSE 产生对预测的梯度，再通过矩阵乘法传播到 weight 和 bias。

MSE 对残差求平方，大误差会获得更大的梯度，因此对离群点敏感。若数据含重尾噪声或异常值，可比较 `L1Loss` 或 `HuberLoss`，但它们对应不同的误差假设和优化性质。特征尺度也会改变损失曲面的曲率：不同量纲相差很大时，同一学习率难以让所有方向都快速稳定收敛，通常应标准化特征。

全批量梯度是当前数据集目标的精确梯度，但每轮成本随样本数增长；mini-batch 梯度更嘈杂，却能降低单步内存和计算压力，并利用 GPU 的批处理能力。这里选择全批量，是为了让学习率、梯度和参数收敛关系尽可能清晰。

## 官方教程补充

**对应官方源文件：** `beginner_source/pytorch_with_examples.rst`、`beginner_source/examples_nn/polynomial_optim.py`、`beginner_source/examples_autograd/polynomial_autograd.py`

官方多项式回归示例逐层展示 Tensor 手写更新、Autograd、`nn.Module` 和 optimizer 的边界。线性回归同样遵循 `prediction -> scalar loss -> gradients -> update`；`MSELoss` 要求预测与标签 shape 一致，广播虽可能运行却会改变问题。用已知合成参数做恢复实验，可以同时验证数据、梯度方向和训练循环。

**验证练习：** 找到上面源文件中的对应 API，先写出输入、输出和状态变化，再运行本 notebook 的相关实验；如果行为不同，优先检查本地 PyTorch 版本、设备能力和输入契约。

<!-- official-pytorch-supplement-v1 -->

## 检查点

不运行代码，先回答再验证：

1. `x` 和 `y` 为什么都使用 `(128, 1)`，而不是一个用 `(128,)`？
2. 写出 MSE 对 weight 和 bias 的梯度；当所有预测都比标签高时，bias 梯度符号是什么？
3. 为什么真实参数是 3 和 2，训练结果却不必精确等于它们？
4. `losses.append(loss)` 与保存 `loss.item()` 有什么内存差异？
5. 为什么 `losses[-1]` 不是最后一次 `optimizer.step()` 之后重新计算的损失？
6. 从 epoch 20 的 checkpoint 使用 `--epochs 40 --resume ...`，实际会执行多少轮更新？

## 试一试

1. **比较学习率**：分别用 `2.0`、`0.2` 和 `0.002` 从相同初始化训练，绘制三条损失曲线，解释发散、稳定收敛和慢收敛。每组实验必须重建模型与 optimizer。
2. **检查解析解**：给 x 添加一列全 1，用 `torch.linalg.lstsq` 求 weight 和 bias，与 SGD 结果比较。
3. **加入离群点**：把少数 y 改成极端值，分别使用 MSE、L1 和 Huber loss，比较恢复参数及残差。
4. **改成 mini-batch**：用 `TensorDataset` 和 DataLoader 训练，比较 batch size 为 8、32、128 时损失曲线的噪声与每轮更新次数。
5. **验证恢复等价性**：比较连续训练 40 轮和训练 20 轮后恢复到 40 轮的最终参数；固定数据、初始化和状态以保证公平比较。

## 常见错误与调试

- **遗漏 `unsqueeze(1)`**：预测可能是 `(N, 1)`、标签却是 `(N,)`，广播成 `(N, N)` 并产生错误目标，MSE 通常会发出 shape 警告。训练前断言 prediction 与 target 形状相同。
- **记录带图的 loss 张量**：列表会持续引用计算图并增加内存。日志保存 `loss.item()`，需要 Tensor 时使用 `loss.detach()`。
- **忘记清梯度**：PyTorch 默认累积 `.grad`，使更新幅度随轮次异常变化。每个独立更新前清梯度。
- **比较学习率时复用已训练模型**：各曲线起点不同，结论无效。固定种子并为每次实验重建模型和 optimizer。
- **学习率过高或输入尺度过大**：loss 可能振荡、变成 `inf`/`nan`。降低学习率、标准化特征并检查梯度是否有限。
- **只恢复模型、不恢复 optimizer**：对当前无动量 SGD 影响有限，但对 momentum SGD/Adam 会改变后续轨迹。始终按训练状态恢复。
- **误解 `--epochs`**：恢复脚本把它当最终 epoch 编号，不是额外轮数。检查 checkpoint metadata 和打印的起始 epoch。